# Experiment 3: Anode current vs. tube temperature

Analysis of thermionic emission current as a function of mercury vapor pressure (via temperature).

## Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from iminuit import Minuit
from iminuit.util import make_func_code, describe
from sklearn.metrics import r2_score
import warnings
warnings.simplefilter("ignore")

# Copy the regressor classes from Exp2
from abc import ABC, abstractmethod
from matplotlib.offsetbox import AnchoredText

class Regressor(ABC):
    @abstractmethod
    def __init__(self, model : callable, x : list, y: list, dx : list, dy : list)->None:
        pass

    @abstractmethod
    def __call__(self, *par)->float:
        pass

    def goodness_estimator(self, optimizer : callable, method : str):
        self.goodness_methods = ["chi2","mse","mae","r2"]
        y_pred = self.model(self.x, *self.par_values)
        if method not in self.goodness_methods:
            raise ValueError(f"No method of type {method} are available.")
        if method == "mse":
            from sklearn.metrics import mean_squared_error
            return mean_squared_error(y_pred=y_pred,y_true=self.y) , f"MSE = {mean_squared_error(y_pred=y_pred,y_true=self.y):.4f}"
        if method == "r2":
            return r2_score(y_pred=y_pred,y_true=self.y) , f"$R^2$ = {r2_score(y_pred=y_pred,y_true=self.y):.4f}"
        if method == "chi2":
            self.par = optimizer.parameters
            self.chi2 = optimizer.fval
            self.ndof = len(self.x) - len(self.par)
            self.chi_ndof = self.chi2 / self.ndof
            return self.chi_ndof , f"$\chi^2$ / ndof = {self.chi_ndof:.4f}({self.chi2:.4f}/{self.ndof})"

    def show(self, optimizer : callable , x_title="X", y_title="Y", plot_title=None, goodness_loc=2, legend_loc='best', show_grid=False, sig_digits=3, metric="chi2"):
        self.par        = optimizer.parameters
        self.par_values = optimizer.values
        self.par_errors = optimizer.errors
        text=""
        for p in (self.par):
            text += rf"{p} = {self.par_values[p]:.{sig_digits}f} $\pm$ {self.par_errors[p]:.{sig_digits}f}" + "\n"
        self.goodness_metric, metric_text = self.goodness_estimator(optimizer, method=metric)
        text = text + metric_text

        self.func_x = np.linspace(self.x[0],self.x[-1] ,10000)
        self.y_fit = self.model(self.func_x, *self.par_values)

        plt.rc("font", size=16)
        fig = plt.figure(figsize=(8,6))
        ax = fig.add_axes([0,0,1,1])
        if plot_title is not None:
            ax.set_title(plot_title)
        ax.plot(self.func_x, self.y_fit, label="Fit")
        ax.scatter(self.x, self.y, c="red")
        ax.errorbar(self.x, self.y, self.dy, self.dx, fmt='none', ecolor='red', capsize=3, label="Data")
        ax.set_xlabel(x_title, fontdict={"size":21})
        ax.set_ylabel(y_title, fontdict={"size":21})
        ax.tick_params(axis="both", labelsize=18)
        anchored_text = AnchoredText(text, loc=goodness_loc)
        ax.add_artist(anchored_text)
        ax.legend()
        plt.grid(show_grid)

class Chi2Reg(Regressor):
    def __init__(self, model, x, y, dx=None, dy=None):
        self.model = model
        self.x  = np.asarray(x)
        self.y  = np.asarray(y)
        if dx is None:
            self.dx = np.zeros(x.shape)
        else:
            self.dx = np.asarray(dx)
        if dy is None:
            raise ValueError("Uncertainties on y were not provided!")
        self.dy = np.asarray(dy)
        self.func_code = make_func_code(describe(self.model)[1:])
        self.h = (x[-1]-x[0])/10000

    @property
    def ndata(self):
        return len(self.x)

    def __call__(self, *par):
        self.ym = self.model(self.x, *par)
        df = (self.model(self.x + self.h, *par)-self.ym)/self.h
        chi2 = sum(((self.y - self.ym)**2)/(self.dy**2+(df * self.dx)**2))
        return chi2

%matplotlib inline

## Load data

In [ ]:
exp3m1_df = pd.read_csv('exp3m1.csv')
exp3m1_df.rename(columns={'Temp [c]': 'temp_C', 'Current (a.u.)': 'current'}, inplace=True)
exp3m1_df = exp3m1_df.sort_values('temp_C').reset_index(drop=True)

print(f"Temperature range: {exp3m1_df['temp_C'].min():.1f}°C to {exp3m1_df['temp_C'].max():.1f}°C")
print(f"Current range: {exp3m1_df['current'].min():.4f} to {exp3m1_df['current'].max():.4f} a.u.")
print(f"Total points: {len(exp3m1_df)}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(exp3m1_df['temp_C'], exp3m1_df['current'], s=10, alpha=0.6)
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Current (a.u.)', fontsize=12)
ax.set_title('Exp 3: Anode current vs. tube temperature')
ax.grid(True, alpha=0.3)
plt.show()

## Compute vapor pressure and mean free path

In [ ]:
# Antoine equation constants for mercury (standard literature values)
A_antoine = 8.10622
B_antoine = 3054.6
C_antoine = 274.0

def vapor_pressure_Pa(T_C):
    """Vapor pressure from Antoine equation (output in Pa)."""
    log10_P_mmHg = A_antoine - B_antoine / (T_C + C_antoine)
    P_mmHg = 10 ** log10_P_mmHg
    return P_mmHg * 133.322  # convert mmHg to Pa

def mean_free_path(T_C, P_Pa):
    """Mean free path λ = (4/√2) × k_B × T / (P × π × R_0²)."""
    k_B = 1.381e-23  # J/K
    R_0 = 155e-12    # m (Hg atom radius)
    T_K = T_C + 273.15
    lam = (4 / np.sqrt(2)) * k_B * T_K / (P_Pa * np.pi * R_0**2)
    return lam

exp3m1_df['temp_K'] = exp3m1_df['temp_C'] + 273.15
exp3m1_df['P_Pa'] = exp3m1_df['temp_C'].apply(vapor_pressure_Pa)
exp3m1_df['lambda_m'] = exp3m1_df.apply(lambda row: mean_free_path(row['temp_C'], row['P_Pa']), axis=1)
exp3m1_df['lambda_mm'] = exp3m1_df['lambda_m'] * 1000

print("Sample data with computed quantities:")
print(exp3m1_df[['temp_C', 'temp_K', 'P_Pa', 'lambda_mm', 'current']].head(10))

## Select exponential regime (non-saturated)

In [ ]:
# Exclude saturated region (high-T plateau with I ≈ 7.26)
# Keep only points where I < 6 to avoid Child-Langmuir saturation
saturation_threshold = 6.0
exp3_fit = exp3m1_df[exp3m1_df['current'] < saturation_threshold].copy()

print(f"Excluded {len(exp3m1_df) - len(exp3_fit)} saturated points")
print(f"Fitting on {len(exp3_fit)} points with I < {saturation_threshold}")
print(f"Temperature range of fit region: {exp3_fit['temp_C'].min():.1f}°C to {exp3_fit['temp_C'].max():.1f}°C")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(exp3m1_df['temp_C'], exp3m1_df['current'], s=10, alpha=0.4, label='All data')
ax.scatter(exp3_fit['temp_C'], exp3_fit['current'], s=10, alpha=0.8, color='red', label=f'Fit region (I < {saturation_threshold})')
ax.axhline(saturation_threshold, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Current (a.u.)', fontsize=12)
ax.set_title('Exponential regime selection')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Fit I = a × exp(b/λ)

In [ ]:
# Model: I = a × exp(b/λ)
exp_model = lambda lam, a, b: a * np.exp(b / lam)

# Prepare data for fit
x_fit = exp3_fit['lambda_m'].to_numpy()  # mean free path in meters
y_fit = exp3_fit['current'].to_numpy()
dy_fit = np.full_like(y_fit, 0.05)  # assume ~5% uncertainty on current
dx_fit = np.full_like(x_fit, 0.1 * x_fit)  # ~10% uncertainty on λ

# Chi2 regression
chi2_reg = Chi2Reg(exp_model, x_fit, y_fit, dy=dy_fit, dx=dx_fit)
mopt = Minuit(chi2_reg, a=0.1, b=-0.005)
mopt.migrad()

# Fit results
a_val = mopt.values['a']
a_err = mopt.errors['a']
b_val = mopt.values['b']
b_err = mopt.errors['b']
chi2_ndof = mopt.fval / (len(x_fit) - mopt.nfit)

print(f"\nFit results:")
print(f"  a = {a_val:.6f} ± {a_err:.6f}  (Child-Langmuir prefactor)")
print(f"  b = {b_val:.6f} ± {b_err:.6f}  m  (characteristic length scale)")
print(f"  b = {b_val*1e3:.4f} ± {b_err*1e3:.4f}  mm")
print(f"  χ²/ndof = {chi2_ndof:.4f}")
print(f"\nPhysical interpretation:")
print(f"  |b| ≈ {abs(b_val)*1e3:.2f} mm (grid-to-anode distance)")

## Visualize fit

In [ ]:
# Plot I vs λ with fit
x_plot = np.linspace(x_fit.min(), x_fit.max(), 500)
y_plot = exp_model(x_plot, a_val, b_val)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x_fit*1e6, y_fit, s=40, alpha=0.6, color='red', label='Data (fit region)', zorder=3)
ax.plot(x_plot*1e6, y_plot, 'b-', linewidth=2.5, label='Fit: $I = a \\times e^{b/\\lambda}$', zorder=2)

# Add parameter box
text_box = f"""$a$ = {a_val:.4f} ± {a_err:.4f}
$b$ = {b_val*1e3:.3f} ± {b_err*1e3:.3f} mm
$\\chi^2$/ndof = {chi2_ndof:.3f}"""
ax.text(0.98, 0.97, text_box, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('Mean free path  λ  (μm)', fontsize=12)
ax.set_ylabel('Current  I  (a.u.)', fontsize=12)
ax.set_title('Thermionic emission current vs. mean free path', fontsize=13)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Residual analysis

In [ ]:
y_pred = exp_model(x_fit, a_val, b_val)
residuals = y_fit - y_pred
residual_std = np.std(residuals, ddof=2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs λ
ax1.scatter(x_fit*1e6, residuals, s=40, alpha=0.6, color='purple')
ax1.axhline(0, color='k', linestyle='-', linewidth=0.8)
ax1.axhline(residual_std, color='r', linestyle='--', alpha=0.5, label=f'±1σ = ±{residual_std:.4f}')
ax1.axhline(-residual_std, color='r', linestyle='--', alpha=0.5)
ax1.set_xlabel('Mean free path  λ  (μm)', fontsize=11)
ax1.set_ylabel('Residual  I - Î  (a.u.)', fontsize=11)
ax1.set_title('Residuals vs. λ')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Histogram
ax2.hist(residuals, bins=15, alpha=0.7, color='purple', edgecolor='k')
ax2.axvline(0, color='r', linestyle='-', linewidth=1.5, label='Mean = 0')
ax2.set_xlabel('Residual (a.u.)', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title(f'Residual distribution (σ = {residual_std:.4f})')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()